In [91]:
import pprint
import glob
import random
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from shapely.geometry import Polygon

Process FAOSTAT data so that the commodity expansion rates can be produced for any period between 2000-2020


### Steps
- process FAOSTAT data to have values for every year 2000-2020
- load some forestloss patches
- get the time range 
- use it to query the tabular data

## Process FAOSTAT data (incl. normalization)

In [92]:
df = pd.read_csv('../data/tmp/faostat_data_per_year.csv')

In [93]:
df.Item.unique()

array(['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Oil palm fruit', 'Natural rubber in primary forms'], dtype=object)

In [94]:
FAOSTAT_classes = ['Cattle', 'Oil palm fruit', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']

In [95]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

In [96]:

world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])


tropical_countries = world.loc[world.intersects(tropics)]

#Equal-area projection
tropical_countries['area'] = \
    tropical_countries.to_crs('EPSG:6933').area / 1000000

/tmp/ipykernel_1830752/3496209450.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')
/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [97]:
names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    tropical_countries.loc[tropical_countries['name'] == name, 'name'] = country_name_mapping[name]
    

In [98]:
df_filtered = df.loc[
    (df.Item == 'Cattle') |
    ((df.Item == 'Oil palm fruit') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Cocoa beans') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Coffee, green') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Natural rubber in primary forms') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Soya beans') & (df.Element == 'Area harvested' )) 
]

In [99]:
# 2.71 Mha - area under cultivation of cocoa for 2021 by Kalischek 

#I should probably interpolate for the earlier years as well, 
#it does not make sense to have a massive jump between 2019 and 2020

#df_filtered.loc[
#    ((df_filtered.Area == 'Ghana') & 
#     (df_filtered.Year == 2020) & 
#     (df_filtered.Item == "Cocoa beans")), 'Value'] = 2710000


In [100]:
df_filtered = df_filtered.rename(columns={'Area': 'country'})

### How many countries have data for each commodity

In [101]:
elements = {
    'Cattle' : {}, 
    'Oil palm fruit' : {}, 
    'Soya beans' : {}, 
    'Cocoa beans' : {}, 
    'Coffee, green' : {}, 
    'Natural rubber in primary forms' : {}
}

for idx, row in tropical_countries.iterrows():
    
    #try:
    #    country_name = country_name_mapping[row['name']]
    #except KeyError:
    country_name = row['name']
    
    _df = df.loc[df.Area == country_name]
    if len(_df) == 0:
        print('No FAOSTAT data found for ', row['name'])
    for _cls in FAOSTAT_classes:
        commodity_df = _df.loc[_df.Item == _cls]
        elements_available = commodity_df.Element.unique()
        
        for element in elements_available:
            try:
                elements[_cls][element] += 1
            except KeyError:
                elements[_cls][element] = 1

No FAOSTAT data found for  W. Sahara
No FAOSTAT data found for  Iran
No FAOSTAT data found for  Somaliland


In [104]:
df_filtered = df_filtered.loc[df_filtered.country.isin(tropical_countries['name'].unique())]

In [114]:
df_filtered.to_csv('../data/tmp/faostat_reports_2000_2020_per_year_norm_v2.csv', index=False)

In [112]:
df_filtered['Value'] =  (df_filtered.Value - df_filtered.Value.mean()) /  df_filtered.Value.std()

/tmp/ipykernel_1830752/2152009231.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Value'] =  (df_filtered.Value - df_filtered.Value.mean()) /  df_filtered.Value.std()


In [113]:
df_filtered

,Domain Code,Domain,Area Code (M49),country,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note,country_area
0,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2000,2000,An,-0.041107,E,Estimated value,NaN,652285.423048
1,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2001,2001,An,-0.079818,A,Official figure,NaN,652285.423048
2,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2002,2002,An,0.007357,A,Official figure,NaN,652285.423048
3,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2003,2003,An,0.014136,A,Official figure,NaN,652285.423048
4,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2004,2004,An,-0.005784,A,Official figure,NaN,652285.423048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24346,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2016,2016,ha,-0.211180,A,Official figure,NaN,376272.994543
24349,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2017,2017,ha,-0.212157,A,Official figure,NaN,376272.994543
24352,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2018,2018,ha,-0.211336,A,Official figure,NaN,376272.994543
24355,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2019,2019,ha,-0.211584,A,Official figure,NaN,376272.994543


## Normalize by country

In [105]:
areas = []
for idx, row in df_filtered.iterrows():
    _name = row.country
    area = tropical_countries.loc[tropical_countries['name'] == _name]['area'].item()
    areas.append(area)


In [106]:

df_filtered['country_area'] = areas

/tmp/ipykernel_1830752/2276698506.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['country_area'] = areas


In [58]:
def normalize_by_col(df, cols_to_normalize, norm_column):
    """For each column in df from cols_to_normalize, divide by the value in norm_column. Return normalized df"""
    df_normalized = df.copy()
    for col in cols_to_normalize:
        df_normalized[col] = df_normalized[col] / df_normalized[norm_column]
    return df_normalized

In [67]:

df_filtered = normalize_by_col(df_filtered, ['Value'], 'country_area')

## Normalize to range

In [68]:
cattle_values = df_filtered.loc[df_filtered.Element == 'Stocks'].Value
area_prod = df_filtered.loc[df_filtered.Element == 'Area harvested'].Value

#y = (x - mean) / standard_deviation

#Normalize separately cattle and the other production classes 
for i, values in enumerate([cattle_values, area_prod]):

    nanmean = np.nanmean(values)
    nanstd = np.nanstd(values)
    
    #Normalize
    normed_values = (values - nanmean) / nanstd
    
    #Avoid having negative values (0 often symbolizes missing values; there should be no negative values)
    normed_values = normed_values - normed_values.min()
    
    if i == 0:
        df_filtered.loc[df_filtered.Element == 'Stocks', "Value"] = normed_values
    elif i == 1:
        df_filtered.loc[df_filtered.Element == 'Area harvested', "Value"] = normed_values






In [71]:
df_filtered.to_csv('../data/tmp/faostat_reports_2000_2020_per_year.csv', index=False)

### Compute tabular values from TMF patches

then test and compare with static vals

In [77]:
labels = pd.read_csv('/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv')

forestloss_dir = '/home/jan/Documents/EPFL/project_2/data/forestloss_tiles/dataset_v4'
all_patches = glob.glob(forestloss_dir + "/*")

#while True:
for i in range(50):
    row = labels.sample(1)
    forestloss_path = os.path.join(forestloss_dir, f'{row.sampleid.values[0]}.tif')
    with rasterio.open(forestloss_path) as src:
        data = src.read()

    assert np.all(np.mod(data, 1) == 0)
    years = data[(data >= 2000) & (data <= 2020)]
    _min, _max = years.min(), years.max()
    
    print(_min, _max)
    print(np.percentile(years.flatten(), 10), np.percentile(years.flatten(), 90))
    print()
    continue
    country_data = df_filtered.loc[df_filtered.country == row.country.values[0]]
    
    values = []
    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_min)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_max)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)
    

2000.0 2013.0
2000.0 2003.0

2001.0 2020.0
2013.0 2017.0

2000.0 2020.0
2000.0 2007.0

2000.0 2020.0
2003.0 2016.0

2000.0 2020.0
2007.0 2018.0

2007.0 2020.0
2010.0 2019.0

2005.0 2020.0
2009.0 2019.0

2000.0 2016.0
2000.0 2014.0

2016.0 2018.0
2016.0 2018.0

2000.0 2014.0
2002.0 2011.0

2009.0 2019.0
2009.0 2011.0

2000.0 2019.0
2002.0 2019.0

2013.0 2015.0
2013.2 2015.0

2001.0 2004.0
2001.0 2004.0

2001.0 2015.0
2011.0 2012.0

2000.0 2020.0
2008.0 2020.0

2000.0 2016.0
2000.0 2014.0

2000.0 2019.0
2002.0 2015.0

2000.0 2020.0
2005.0 2016.0

2000.0 2019.0
2002.0 2015.0

2012.0 2019.0
2012.0 2017.0

2000.0 2019.0
2002.0 2004.0

2000.0 2018.0
2000.0 2001.0

2013.0 2020.0
2015.0 2019.0

2002.0 2020.0
2012.0 2018.0

2001.0 2020.0
2002.0 2015.0

2000.0 2020.0
2003.0 2016.0

2000.0 2020.0
2002.0 2013.0

2000.0 2020.0
2008.0 2017.0

2000.0 2020.0
2011.0 2019.6

2004.0 2020.0
2004.0 2020.0

2008.0 2020.0
2009.0 2020.0

2010.0 2013.0
2010.0 2010.0

2000.0 2020.0
2012.0 2020.0

2002.0 2019.0


In [75]:
_min, _max

(2002.0, 2020.0)

In [146]:
values = []
for _class in FAOSTAT_classes:
    try:
        production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_min)), 'Value' ].item()
        production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(_max)), 'Value' ].item()
        expansion = production_at_the_end - production_at_the_start
    except ValueError: #Signifies that one (or both) of the values are missing
        expansion = 0
    values.append(expansion)

